# 14 — Collision Handling in Hash Tables

## Learning goals
- Understand why hash collisions happen.
- Learn how hash tables handle collisions.
- Compare separate chaining and open addressing.
- Understand linear probing, quadratic probing, and double hashing.
- Learn load factor and resizing.
- Connect the theory to Python dictionaries and sets.

**Important:** Python's built-in `dict` and `set` handle collisions internally. You normally should not implement a hash table yourself in application code. This concept helps you understand performance and interview fundamentals.

## 1. What is a hash collision?

A hash function maps keys to hash values. A hash table maps those hash values to a finite set of storage locations.

A collision occurs when two different keys map to the same table index.

For example, if a simple table uses:

`index = key % 10`

then keys `12` and `22` both map to index `2`.

A collision is normal. It does not mean the hash table is broken.

In [1]:
def simple_hash_index(key: int, table_size: int) -> int:
    return key % table_size

print(simple_hash_index(12, 10))  # 2
print(simple_hash_index(22, 10))  # 2
print(simple_hash_index(32, 10))  # 2

2
2
2


## 2. Why do collisions happen?

Common reasons:
- The number of possible keys is much larger than the number of table slots.
- A hash function distributes keys unevenly.
- Different keys can produce the same hash value.

By the pigeonhole principle, if there are more possible keys than slots, at least two keys must share a slot.

A good hash function aims for a uniform distribution, but cannot guarantee that collisions never happen.

## 3. Collision strategy A — Separate chaining

Each table slot stores a collection (often a linked list or dynamic array) of key-value pairs.

When multiple keys map to the same index, they are stored in that slot's collection.

Lookup:
1. Compute the index.
2. Visit the collection at that index.
3. Compare keys until the matching key is found.

In [2]:
class ChainingHashTable:
    def __init__(self, size=10):
        self.size = size
        self.buckets = [[] for _ in range(size)]

    def _index(self, key):
        return hash(key) % self.size

    def put(self, key, value):
        bucket = self.buckets[self._index(key)]

        for i, (old_key, old_value) in enumerate(bucket):
            if old_key == key:
                bucket[i] = (key, value)
                return

        bucket.append((key, value))

    def get(self, key):
        bucket = self.buckets[self._index(key)]

        for stored_key, value in bucket:
            if stored_key == key:
                return value

        raise KeyError(key)

table = ChainingHashTable(size=5)
table.put("apple", 10)
table.put("banana", 20)
table.put("orange", 30)

print(table.get("apple"))
print(table.get("banana"))

10
20


## 4. Separate chaining — complexity

Let:
- `n` = number of stored keys.
- `m` = number of buckets.
- `alpha = n / m` = load factor.

With a good hash function and a reasonable load factor:
- Expected lookup: `O(1 + alpha)`, often treated as expected `O(1)` when alpha is bounded.
- Expected insertion: `O(1)` under the same assumption.
- Worst case: `O(n)` if all keys land in one bucket.

Space: `O(n + m)`.

## 5. Collision strategy B — Open addressing

In open addressing, all entries are stored directly in the table array.

If the target slot is occupied, the algorithm probes other slots according to a probe sequence until it finds:
- The key,
- An empty slot, or
- The end of the search sequence.

Unlike chaining, there is no separate bucket collection.

## 6. Linear probing

Try the next slot one by one:

`index = (hash(key) + i) % table_size`

where `i = 0, 1, 2, ...`.

Advantages:
- Simple.
- Good cache locality.

Disadvantage:
- Primary clustering: long groups of occupied slots can form, making future probes longer.

In [3]:
class LinearProbingSet:
    def __init__(self, capacity=8):
        self.capacity = capacity
        self.table = [None] * capacity

    def _index(self, key):
        return hash(key) % self.capacity

    def add(self, key):
        start = self._index(key)

        for step in range(self.capacity):
            i = (start + step) % self.capacity

            if self.table[i] == key:
                return

            if self.table[i] is None:
                self.table[i] = key
                return

        raise OverflowError("Hash table is full")

    def contains(self, key):
        start = self._index(key)

        for step in range(self.capacity):
            i = (start + step) % self.capacity

            if self.table[i] is None:
                return False

            if self.table[i] == key:
                return True

        return False

example = LinearProbingSet(5)
for value in [1, 6, 11]:
    example.add(value)

print(example.table)
print(example.contains(6))   # True
print(example.contains(16))  # False

[None, 1, 6, 11, None]
True
False


## 7. Deletion in open addressing — tombstones

A subtle issue: if we simply change a deleted slot to empty, a search may stop there and incorrectly conclude that a key later in the probe sequence is absent.

A common solution is a **tombstone**:
- `None` means a slot has never been occupied; search can stop here.
- A special deleted marker means the slot was used but is now available; search must continue.

This simple teaching implementation demonstrates tombstones.

In [4]:
class TombstoneSet:
    DELETED = object()

    def __init__(self, capacity=8):
        self.capacity = capacity
        self.table = [None] * capacity

    def _index(self, key):
        return hash(key) % self.capacity

    def add(self, key):
        start = self._index(key)
        first_deleted = None

        for step in range(self.capacity):
            i = (start + step) % self.capacity
            slot = self.table[i]

            if slot == key:
                return

            if slot is self.DELETED and first_deleted is None:
                first_deleted = i

            if slot is None:
                target = first_deleted if first_deleted is not None else i
                self.table[target] = key
                return

        if first_deleted is not None:
            self.table[first_deleted] = key
            return

        raise OverflowError("Hash table is full")

    def contains(self, key):
        start = self._index(key)

        for step in range(self.capacity):
            i = (start + step) % self.capacity
            slot = self.table[i]

            if slot is None:
                return False

            if slot is not self.DELETED and slot == key:
                return True

        return False

    def discard(self, key):
        start = self._index(key)

        for step in range(self.capacity):
            i = (start + step) % self.capacity
            slot = self.table[i]

            if slot is None:
                return

            if slot is not self.DELETED and slot == key:
                self.table[i] = self.DELETED
                return

example = TombstoneSet(5)
for value in [1, 6, 11]:
    example.add(value)

example.discard(6)
print(example.contains(11))  # True: search passes the tombstone

True


## 8. Quadratic probing

Instead of checking consecutive slots, use a quadratic offset:

`index = (hash(key) + c1*i + c2*i*i) % table_size`

This can reduce primary clustering compared with linear probing.

However, the chosen table size and constants matter. A probe sequence may fail to visit every slot, so a table can appear full even when some slots are empty.

## 9. Double hashing

Use a second hash function to determine the step size:

`index = (h1(key) + i * h2(key)) % table_size`

A good second hash must produce a non-zero step that is relatively prime to the table size, so the probe sequence can visit every slot.

Double hashing usually reduces clustering compared with linear probing.

## 10. Load factor

Load factor measures how full a hash table is:

\[
\alpha = \frac{number\ of\ stored\ entries}{table\ capacity}
\]

For open addressing, performance often degrades quickly as the load factor approaches `1`.

For chaining, load factor can be greater than `1`, because each bucket can hold multiple entries.

In [5]:
def load_factor(entries: int, capacity: int) -> float:
    if capacity <= 0:
        raise ValueError("capacity must be positive")
    return entries / capacity

print(load_factor(7, 10))  # 0.7
print(load_factor(10, 10)) # 1.0

0.7
1.0


## 11. Resizing and rehashing

When a table becomes too full, it can:
1. Allocate a larger backing array.
2. Recompute each key's new index using the new capacity.
3. Insert all entries into the new table.

This is called resizing/rehashing.

Why recompute? The table index depends on capacity, so old positions generally cannot just be copied unchanged.

A resize costs `O(n)` for `n` entries, but if resizing happens occasionally (for example, when crossing a load-factor threshold), insertion can still have amortized `O(1)` cost.

## 12. Python dictionaries and sets

Python's `dict` and `set` are hash-table-based structures. They handle:
- Hashing.
- Collision resolution.
- Resizing.
- Key equality checks.

You should rely on the built-in structures in normal Python solutions.

Important rules:
- Equal objects must have equal hash values.
- A key's hash must remain stable while it is in a dictionary or set.
- A collision does not imply key equality.
- Dictionary operations are expected `O(1)` on average; worst-case behavior can be worse.

In [6]:
# Different keys can coexist even if their hash values collide.
class SameHash:
    def __init__(self, value):
        self.value = value

    def __hash__(self):
        return 42

    def __eq__(self, other):
        return isinstance(other, SameHash) and self.value == other.value

data = {SameHash("x"): 1, SameHash("y"): 2}
print(len(data))  # 2
print(data[SameHash("x")])  # 1

2
1


## 13. Chaining vs open addressing

| Feature | Separate chaining | Open addressing |
|---|---|---|
| Where entries live | Bucket collections | Table array itself |
| Load factor | Can exceed 1 | Must stay below 1 |
| Deletion | Usually straightforward | Needs tombstones or careful shifting |
| Clustering | Bucket chains can grow | Probe clustering can occur |
| Cache locality | Often less contiguous | Often good |
| Full table | Buckets can still accept entries | Must resize or fail |

Both approaches can provide expected constant-time operations with suitable hashing and table management.

## 14. Complexity summary

| Operation | Expected | Worst case |
|---|---:|---:|
| Search | `O(1)` | `O(n)` |
| Insert | `O(1)` amortized | `O(n)` |
| Delete | `O(1)` expected | `O(n)` |
| Resize / rehash | `O(n)` | `O(n)` |

The expected bounds rely on a good distribution and controlled load factor. A resize is individually linear, but occasional resizing can preserve amortized constant-time insertion.

## 15. Common interview mistakes

1. Thinking collisions mean two keys are equal.
2. Assuming a hash function can prevent all collisions.
3. Forgetting to compare keys after locating a candidate slot.
4. Stopping a search at a tombstone.
5. Using an open-addressing table at an excessively high load factor.
6. Forgetting to rehash entries after capacity changes.
7. Claiming every dictionary operation is guaranteed worst-case `O(1)`.
8. Implementing a custom table in a coding problem when Python `dict` or `set` is the intended tool.

## 16. Interview recognition checklist

If asked about hash-table internals, explain:
1. What a collision is.
2. Why collisions are unavoidable in finite tables.
3. Separate chaining.
4. Open addressing and probing.
5. Tombstones for deletion.
6. Load factor and resizing.
7. Expected versus worst-case complexity.
8. Why equality checking is necessary after hashing.

If asked to solve a normal DSA problem, use Python `dict` or `set` unless the question explicitly asks you to implement a hash table.

## 17. Practice

1. Implement a hash map from scratch: [706. Design HashMap](https://leetcode.com/problems/design-hashmap/)
2. Implement a hash set from scratch: [705. Design HashSet](https://leetcode.com/problems/design-hashset/)
3. [1206. Design Skiplist](https://leetcode.com/problems/design-skiplist/) — useful for comparing data-structure design approaches.

For 706 and 705, first implement a simple bucket-array solution using separate chaining. Then explain how open addressing differs.

## 18. Final revision

- Collision: different keys map to the same slot.
- Chaining: store multiple entries in one bucket.
- Open addressing: probe for another slot.
- Linear probing: consecutive slots.
- Quadratic probing: quadratic offsets.
- Double hashing: second hash determines the step.
- Tombstone: deleted marker that does not terminate a search.
- Load factor: entries divided by capacity.
- Rehashing: move entries into a resized table.

**Next:** Concept 15 — Rolling Hash (deep dive).